# Replicating the EEG Pre-processing Pipeline

Sample Dataset from:

Tamari Shalamberidze, Kyle Nash, and Jeremy B. Caplan (2026). Resting-State EEG and Trait Anxiety. OpenNeuro. [Dataset] doi: doi:10.18112/openneuro.ds007609.v1.0.0

Note: This dataset uses the BIDS (Brain Imaging Data Structure) framework, which can be easily shared, understood, and reused. With this in mind, I aim to replicate the methodology of this paper by using python libraries, rather than the original EEGLAB toolbox in MATLAB, for the sake scientific replication and practicing independent verification of results.

## Import relevant libraries

In [ ]:
!pip install mne mne-bids numpy matplotlib pandas

import os
import pandas as pd
import numpy as np
import scipy
import matplotlib.pyplot as plt

!pip install mne
import mne

## Step 0: Load and Visualize the Dataset
---


In [ ]:
!pip install mne-bids

In [ ]:
import json
import os

# Ensure the 'data' directory exists
os.makedirs('data', exist_ok=True)

# Create the mandatory BIDS description file
description = {
    "Name": "Trait Anxiety Resting EEG Replication",
    "BIDSVersion": "1.6.0",
    "DatasetType": "raw",
    "Authors": ["Shalamberidze et al."]
}

with open('data/dataset_description.json', 'w') as f:
    json.dump(description, f, indent=4)

In [ ]:
# Use mne-bids library to import the files for sub-1019
## Note: The bids format requires you to store EACH PARTICIPANT's file in a separate folder.
# Hence, the sub-folders sub-1019 and eeg.

from mne_bids import BIDSPath, read_raw_bids
import os

root_path = './data'

bids_path = BIDSPath(
    subject='1019',
    task='rest',
    datatype='eeg', # Adding this helps MNE-BIDS find the right sub-folder
    root=root_path
)

print(f"BIDS path initialized: {bids_path.fpath}")


In [ ]:
from mne_bids import read_raw_bids

# This will read the .set and .fdt files automatically using the BIDS metadata
raw = read_raw_bids(bids_path=bids_path)
raw.load_data()

Compute the power spectral density for this session using the inbuilt mne function ``.compute_psd.plot()``

In [ ]:
raw.compute_psd(fmax=120).plot(picks='data',exclude='bads', amplitude=False)

#Note: If you don't know what a function does, don't shy away from away from running help()

The UserWarning message suggests that channels E91, E229, and E256 are be dead.

This can also be confirmed with the PSD graph showing a comb-like pattern with a ridiculously small power as a result of dB = 10*log(Power) where Power is near 0, so dB approaches -inf. The periodic drops represent a relationship between the fs and lenght of window used for FFT. The details are beyond the scope of this tutorial.

## Step 1.1: Remove dead channels
---

In [ ]:
# Lets compute the PSD again after removing the dead channels!

raw.info['bads']=  ['E91', 'E229', 'E256']
picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
raw.compute_psd(fmax=120).plot(picks='data',exclude='bads', amplitude=False)

Now that the dead channels have been removed, we can start replicating the pre-processing pipeline highlighted in the methodology of this study.

Below is a display of the setup of the EEG-channels.

**NOTE**: Do NOT delete the bad channels yet. They are usually left until the ICA step or interpolated later.

In [ ]:
raw.plot_sensors(show_names=True)

## Step 1.2: Re-referencing
---

The first step in the pipeline is to re-reference the channels to Cz.

But for that, we must first check whether this reference was already made physically before uploading this data!

In [ ]:
#Check the current montage.

print(raw.get_montage())

In [ ]:
#Access the sidecar metadata that MNE-BIDS already parsed.
from mne_bids import get_bids_path_from_fname
import json

# This step changes the target from the 'data' file (.fdt) to the 'metadata' file (.set)
## This is because the get_bids_path_from_fname() doesn't take .fdt (raw data) as an input.
fname_str = str(raw.filenames[0]).replace('.fdt', '.set')

bids_path = get_bids_path_from_fname(fname_str)

# .json file is the human-readable metadata (aka "sidecar")
sidecar_json = bids_path.copy().update(suffix='eeg', extension='.json')

# Verify reference electrode
with open(sidecar_json.fpath, 'r', encoding='utf-8') as f:
    data = json.load(f)
    recorded_ref = data.get('EEGReference')

print(f'The recorded reference in the metadata is {recorded_ref}')

# Rename E129 to Cz so it matches the metadata label
raw.rename_channels({'E129': 'Cz'})

# Load data !
raw.load_data()

# Set reference to recorded_ref (even though in this case it has already been done for you)
raw.set_eeg_reference(ref_channels=[recorded_ref])


## Step 1.3 = Bandpass filter: 0.1Hz-50Hz

---

In [ ]:
raw_filtered = raw.filter(0.1, 50)

Notice the changes in the data!

In [ ]:
raw_filtered.compute_psd(fmax=120).plot(picks='data',exclude='bads', amplitude=False)

The power spectral density of the reference channel (Cz) drop to the floor given that there is 0 variance between Cz and reference [since they are the same], which results in zero power across all frequencies.

In [ ]:
# Remove the Cz and bad channels before bandpass filtering

picks = [ch for ch in raw.ch_names if ch not in raw.info['bads'] and ch != 'Cz']
raw.compute_psd(fmax=70 , picks=picks).plot()

The filter successfully removed frequencies above 50Hz and below 0.1Hz. By default, there is an upper transition bandwidth of 12.5Hz, which is why it is still possible to see the attenuating signals between 50-62.5Hz

The peak at 60Hz is the line noise. It can be removed by applying a Notch filter.

## Step 1.4: Notch filter to remove line noise

---

The original methodology used CleanLine (EEGLAB). However, since we have already applied a lowpass filter at 50Hz, the line noise has already been removed. However, to remain consistent with the pipeline, we apply a notch filterat 60Hz and 120Hz.

In [ ]:
raw.notch_filter(freqs = [60, 120])

one-pass, zero-phase, non-casual = Filter does not shift the waves forward nor backward in time. So analysis of brain timing won't suffer.

One contiguous filter = suppresses everywhere equally.

lower/upper transition bandwidth = precision of the filter

In [ ]:
raw.compute_psd(fmax=70 , picks=picks).plot()

## Step 1.5: Automatic channel rejection by Kurtosis
---

We can remove channels that contain excessive noise or non-biological sources of data by measuring the kurtosis.

kurtosis = "tailedness" of a distribution.

Kurtosis values are standarized by converting to z-score (mean 0, unit standard deviation). A common rejection threshold is 2 standard deviations, although raw values above 5 are immediate outliers.

We aren't interested in _which_ frequencies are present in the channel, rather we are looking at _how often_ a channel displays a voltage that jumps from the normal distribution to an extreme outlier. Thi catches the 'electrode pops' that could otherwise skew analysis.

MNE actually does *not* contain an automatic channel rejection by kurtosis function the way that EEGLAB does. Rather, we will build a kurtosis filter from scratch using first principles.

But first, it is always useful to visualize the distribution of the data before applying any filters.

In [ ]:
from scipy.stats import kurtosis
import numpy as np

picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
data = raw.get_data(picks=picks)
k_vals = kurtosis(data, axis=1, fisher=True) #axis=1, time-domain

fig, ax = plt.subplots(figsize=(10,6))
ax.hist(k_vals, bins=100, color='skyblue', edgecolor = 'black', alpha=0.7)

ax.set_title("Pre-Processing: Distribution of Channel Kurtosis", fontsize=14)
ax.set_ylabel("Number of Channels", fontsize=12)
ax.set_yscale('log')
ax.legend()
ax.grid(axis='y',linestyle='-', alpha=0.6)

plt.tight_layout()
plt.show()

As evident by the above histogram, there are significant outliers in our dataset.

In EEGLAB, "Threshold=2" represents a relative Z-score rather than a raw kurtosis threshold. Using 2.0 as a threshold results in almost all the channels being rejected.

To reproduce the methodology properly, a two-pass approach was performed:

a. Removing the extreme outliers (k_value > 20) to normalize the variance of the distribution.
b. Applying the paper's Z>2.0 threshold to identify subtle outliers.

In [ ]:
import numpy as np
from scipy.stats import kurtosis, zscore

# PASS 1: Absolute Threshold
current_picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
data = raw.get_data(picks=current_picks)
ch_names = [raw.ch_names[p] for p in current_picks]

k_vals_1 = kurtosis(data, axis=1, fisher=True)

# We use an absolute threshold of 20 to filter the channels with outlier voltage
monsters = [name for name, k in zip(ch_names, k_vals_1) if abs(k) > 20.0]
raw.info['bads'].extend(monsters)
print(f"Pass 1 (Absolute): Removed {len(monsters)} outliers.")

# PASS 2: Z-Score Threshold
# Re-pick now that the extreme outliers are excluded
current_picks = mne.pick_types(raw.info, eeg=True, exclude='bads')
data_2 = raw.get_data(picks=current_picks)
ch_names_2 = [raw.ch_names[p] for p in current_picks]

k_vals_2 = kurtosis(data_2, axis=1, fisher=True)
k_zscores = zscore(k_vals_2)

# Now we use the paper's Z-score threshold of 2.0
outliers = [name for name, z in zip(ch_names_2, k_zscores) if abs(z) > 2.0]
raw.info['bads'].extend(outliers)
print(f"Pass 2 (Z-score): Removed {len(outliers)} outliers.")

## Step 1.6: Manual channel rejection by visual inspection and spectral properties
---------------------------------------

Note that during manual inspection of spectral properties, we must first exclude the reference electrode (Cz) since its relative voltage is 0. Including this in the analysis  would result in the PSD having a negative infinity floor and renders the rest of the data uninterpretable.

In [ ]:
exclude = raw.info['bads']+['Cz']
picks = mne.pick_types(raw.info, eeg=True, exclude=exclude)
raw.compute_psd(picks=picks, fmax=60).plot()

Clearly, the previous steps have managed to remove quite a lot of noise from our data. However, there is still information within our recording that acts more as noise than signal for the behavior we are trying to study.

When performing manual rejection, researchers typically look for:

1. Muscle artifacts: High flat "plateau" in the 20-50Hz range.
2. Short artifact: Two channels overlap perfectly.
3. Dead artifact: Straight line with low power.

For which, the channels are rejected if the sensor doesn't work, is short, or the muscle artifact obscures the brain signal.

So first, we will have to identify the channels that show an excessively large power value corresponding to a muscle or external artifact.

To perform this manual rejection , I will use the threshold of 20 dB/Hz for the frequencies between 20-40Hz to reject channels capturing excessively high power frequencies.

In [ ]:
psd = raw.compute_psd(picks='eeg', fmin=20, fmax=40, exclude='bads')
psd_data = psd.get_data()

print(f"Max value in the whole array: {psd_data.max()}")
print(f"Min value in the whole array: {psd_data.min()}")

Now that we know that the largest value in this array is 2.16e-07 V^2/Hz. To make this meaningful and comparable to the visual plots earlier, we have to first convert it to dB and then threshold to reject the EMG-inflicted channel.


In [ ]:
import numpy as np

psd = raw.compute_psd(picks='eeg', fmin=20, fmax=40, exclude=exclude)
psd_data, frequencies = psd.get_data(return_freqs=True) #V^2/Hz

psd_db = 10 * np.log10(psd_data * 1e12) #V^2 to uV^2

mean_power_db = psd_db.mean(axis=1) #average across 20-40Hz frequency bins

#Threshold
threshold = 20
muscle_indices = np.where(mean_power_db > threshold)[0] #Mean power more than 20dB
muscle_ch_names = [psd.ch_names[i] for i in muscle_indices]

print(f"Found {len(muscle_ch_names)} channels with high EMG noise.")

#Add these EMG inflicted channels into the bad list
raw.info['bads'].extend(muscle_ch_names)

## Step 1.7: Average re-reference
---------------------------------------

So far, we have been using Cz as the reference electrode to remove any bad channels, filter, and reject artifacts.

However, given that the Cz channel also has its own biological noise and local brain activity being captured, this relative activity is injected into every other channel in the dataset.

So, we transition to a Global Average Reference, which uses only the channels that have NOT been labelled as "bad" so far to calculate the mean and then subtracting that value from every other channel.

This allows us to treat the scalp as a closed electrical system where total sum of current is 0, making it unbiased for our future topographical analysis (like ICA).

In [ ]:
# Numpy would require you to calculate the mean for every timepoint, get an array of average, and substract it from every channel.
# MNE has an built-in argument for ref_channels, "average", which handles all this math in the background.

#Create a safety copy since set_egg_reference is an in-place operation.
raw_avg = raw.copy()

#Apply global average reference
raw_avg.set_eeg_reference(ref_channels='average', projection=False) #in-built parameter "average" ignores "bads" and subtracts global average reference from each channnel.

#Visualize
raw_avg.compute_psd(fmax=60).plot()

Given that we re-referenced to average, the data has lost one degree of freedom. ('Rank N-1')

So, when performing the ICA decomposition, the algorithm will automatically detect this rank deficiency and ignore it to focus on the 20-30 strongest components.

## Step 1.8 and 1.9: ICA decomposition, ICLabel
---------------------------------------

In [ ]:
from mne.preprocessing import ICA

#Initialize ICA
ica = ICA(n_components=20, method = 'infomax', random_state=97, fit_params = dict(extended=True))

#Fit the ICA to the data
ica.fit(raw_avg)

#Visualize
ica.plot_components()
ica.plot_sources(raw_avg)

Now that we performed the Independent Component Analysis (with 20 components), the next step is to analyze the spatial topography and power frequencies, so that we get a probability score for each component.

This will allow us to remove the components that are most likely to be resulting from non-brain signals.

In [ ]:
pip install mne-icalabel

In [ ]:
from mne_icalabel import label_components

raw_avg.filter(l_freq=1.0, h_freq=100.0) #Apply bandpass filter 1-100 Hz as required for ICLabel.

ic_labels = label_components(raw_avg, ica, method = 'iclabel')

# Labels and probabilities
labels = ic_labels['labels']
probs = ic_labels['y_pred_proba']

# Print results

for i, (label,prob) in enumerate(zip(labels,probs)):
  print(f'Component {i:03d}: {label:<10} (Probability: {prob: .2f})')

As we can see, the ICLabel was not very effective at identifying sources of biological artifact in this dataset. "Other" referes to a component containing a mix of signals that the Labeler could not cleanly categorize.

This may be due to the model being trained on lower-density caps than our sample of 257 electrodes.

So, lets re-attempt classfication using manual verification.

## Step 1.10 and 1.11: Manual verification and removal of flagged components
---------------------------------------

Looking back at the ICA output in step 1.8 and the label output in 1.9, we can identify some key components:

* Component 012 = Brain, 90%.

* Component 015 = Other, 88%. Localized blob in the topomap suggests EMG activity.

* Components 000/004/009/011: Channel Noise.

* Components 006/007/008: Heart, 30-40%. *BUT* visual inspection of ``ica.plot_sources(raw_avg)`` suggests normal variation rather than ECG artifacts.

In [ ]:
#Based on the ICLabel and ica.plot_sources:

ica.exclude = [0, 4, 9, 11, 15]

raw_cleaned = raw_avg.copy().filter(1,40) #Smaller range for final visualization
ica.apply(raw_cleaned)

## Step 1.12: Interpolation of rejected channels using pre-rejection filtered data
---------------------------------------

In [ ]:
raw_interp = raw_cleaned.copy().interpolate_bads(reset_bads=True, method='spline')

print(f"Remaining bad channels after interpolation: {raw_interp.info['bads']}")

In [ ]:
# Final Output after pre-processing

raw_interp.compute_psd(fmax=60).plot(picks='data', amplitude=False)

In the final output, we removed the 60Hz power line noise and low-passed the data at 40Hz to eliminate muscle artifacts.

While the 'After' plot looks significantly smoother, this represents the removal of non-biological noise, allowing the brain's natural Alpha rhythm to finally emerge as a peak around 10Hz.

# EEG Analysis

The participants of the study were asked to solve a set of anxiety quesstionaires and then spend 1 minute in each of the following states while their EEG was being recorded:

Eyes Open - Eyes Closed - Eyes Open - Eyes Closed

The onset and termination of each block was marked by an auditory cue, which has been stored as an event marker in the raw descriptions and annotations.

In [ ]:
# Extract marker timestamps

onsets = raw.annotations.onset
descriptions = raw.annotations.description

markers = {}
beep_count = 1

for onset, desc in zip(onsets, descriptions):
  if 'bgin' in desc:
    markers['EO1_start'] = onset
  elif 'Beep' in desc:
    markers[f'Beep_{beep_count}'] = onset
    beep_count += 1

print(markers)

The researchers decided to remove 12 seconds from each of the 4 blocks to avoid capturing edge-artifacts and the transient evoked response that took place after each auditory beep.

We have implemented the same slicing logic in the following code block:

In [ ]:
# Slice using the event markers extracted above.

eo_1 = raw.copy().crop(tmin=markers['Beep_1'] + 12.0, tmax=markers['Beep_2'])

ec_1 = raw.copy().crop(tmin=markers['Beep_2'] + 12.0, tmax=markers['Beep_3'])

eo_2 = raw.copy().crop(tmin=markers['Beep_3'] + 12.0, tmax=markers['Beep_4'])

ec_2 = raw.copy().crop(tmin=markers['Beep_4'] + 12.0, tmax=None)

We will now combine the first and second EC epochs linearly since frequency analysis would not be influenced by the temporal sequence of these epochs.

The same will be done for the EO epochs.

In [ ]:
#Combine segments
eo_combined = mne.concatenate_raws([eo_1, eo_2])
ec_combined = mne.concatenate_raws([ec_1, ec_2])

#Calculate PSD for EO and EC
psd_eo = eo_combined.compute_psd(fmin=4, fmax=13, picks=[str('E21'), str('E101')]) #Note: E21 is Fz, E101 is Pz
psd_ec = ec_combined.compute_psd(fmin=4, fmax=13, picks=[str('E21'), str('E101')])

#Extract data
data_eo, freqs = psd_eo.get_data(return_freqs=True)
data_ec, freqs = psd_ec.get_data(return_freqs=True)

Finally, we will extract the alpha and theta band powers for both these conditions (EO and EC) to assess whether there was an increase in average  frequency.

In [ ]:
import numpy as np

# Extract Band Power
theta_mask = (freqs >= 4) & (freqs <= 8)
alpha_mask = (freqs >= 8) & (freqs <= 13)

# Get mean band power for Fz and Pz
fz_alpha_eo = psd_eo.get_data(picks=['E21'])[:, alpha_mask].mean()
fz_alpha_ec = psd_ec.get_data(picks=['E21'])[:, alpha_mask].mean()

pz_alpha_eo = psd_eo.get_data(picks=['E101'])[:, alpha_mask].mean()
pz_alpha_ec = psd_ec.get_data(picks=['E101'])[:, alpha_mask].mean()

# Calculate percentages
fz_diff = ((fz_alpha_ec - fz_alpha_eo) / fz_alpha_eo) * 100
pz_diff = ((pz_alpha_ec - pz_alpha_eo) / pz_alpha_eo) * 100

print("--- FINAL NEUROSCIENCE RESULTS ---")
print(f"Fz (E21) Alpha: EO={fz_alpha_eo:.2e}, EC={fz_alpha_ec:.2e} ({fz_diff:+.1f}% change)")
print(f"Pz (E101) Alpha: EO={pz_alpha_eo:.2e}, EC={pz_alpha_ec:.2e} ({pz_diff:+.1f}% change)")

The results suggest that the eyes closed conditions had an increased proportion of Alpha and Theta frequencies as compared to the eyes open condition.

However, the +660.2% change in Alpha at Fz is likely an anomaly due to muscle or blink artifacts, despite having manually removed some of the sources of noise during the pre-processing phase.

To verify this anomaly, we will plot the absolute alpha at Fz and Pz for comparison.

In [ ]:
# Look at absolute values instead of percentages
print(f"Fz Absolute Alpha (EC): {fz_alpha_ec:.2e}")
print(f"Pz Absolute Alpha (EC): {pz_alpha_ec:.2e}")

# Plot the PSD to see the "shape" of the Alpha peak
import matplotlib.pyplot as plt
psd_ec.plot(picks=['E21', 'E101'])

As evident, the Fz channel shows a drastically elevated alpha power that likely does not correspond to a clean neural oscillation. This is likely due to EMG noise near the forehead or slow eye movements.

However, the results for alpha and theta changes are Pz are more representative of what we observe in the literature during eyes-open and eyes-closed conditions.